# GFQL Cypher Benchmark: CPU/GPU DataFrames vs Neo4j

Run Cypher graph queries and analytics directly on Python dataframes — no database required.
This benchmark compares **Graphistry's local Cypher** (CPU and GPU) against **Neo4j + GDS**
on the same end-to-end graph pipeline.


## TL;DR

| | Neo4j + GDS | Graphistry CPU | Graphistry GPU | GPU speedup vs Neo4j |
|---|---|---|---|---|
| **Twitter** (2.4M edges) | 13.51s | 2.55s | **0.30s** | **45x** |
| **GPlus** (30M edges) | >187s | 75.78s | **3.33s** | **>56x** |

Warm median of 5 runs after 2 warmup iterations on `dgx-spark` (GB10).
Neo4j GPlus is a lower bound — the transaction did not finish.

All three engines run the same pipeline shape:
graph search → PageRank enrichment → graph search.
The Graphistry paths stay entirely in Python dataframes — no external database required.


## What is GFQL?

**GFQL** is Graphistry's dataframe-native graph query language. Instead of sending graph queries to an external database, GFQL runs directly on your in-memory Python graph/dataframe objects.

In this benchmark, the same overall workflow runs on:

- **CPU**: `pandas + igraph`
- **GPU**: `cudf + cugraph`

That is why the comparison is interesting: we are not switching products between CPU and GPU. We are switching execution backends while keeping the workflow dataframe-native.


## What is being benchmarked?

This is an end-to-end graph pipeline, not a single algorithm microbenchmark:

1. **Data loading** from a cached SNAP edge list
2. **Data shaping** to compute node degree as reusable node metadata
3. **Graph search** with local Cypher `GRAPH { MATCH ... }`
4. **Graph analytics** with local Cypher `CALL graphistry.{igraph,cugraph}.pagerank.write()`
5. **Graph search again** with local Cypher `GRAPH { MATCH ... }`
6. **Downstream use** of the final graph for analysis or visualization

That means the benchmark covers dataframe-native wrangling, graph search, and graph analytics in one story.


## Benchmark environment

These results come from **saved DGX runs**, not from rerunning the benchmark in this notebook.

- Host: `dgx-spark`
- GPU: `GB10`
- NVIDIA driver: `580.126.09`
- Container/runtime: `graphistry/test-gpu:latest`
- Result artifacts: `plans/gfql-gpu-pagerank-benchmark/results/`

The point of this notebook is to explain the benchmark clearly, not to make the reader wait for large graph runs.


```python
result = g.gfql(f"""
    GRAPH g1 = GRAPH {{
      MATCH (n)-[e]-(m)
      WHERE n.degree >= $degree_cutoff
    }}
    GRAPH g2 = GRAPH {{
      USE g1
      CALL graphistry.{backend}.pagerank.write()
    }}
    GRAPH {{
      USE g2
      MATCH (n)-[e]-(m)
      WHERE n.pagerank >= $pagerank_cutoff
    }}
""",
    params={
        "degree_cutoff": degree_cutoff,
        "pagerank_cutoff": pagerank_cutoff,
    },
    engine=engine,  # "pandas" for CPU, "cudf" for GPU
)
```


In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from benchmarks.gfql.filter_pagerank import presentation as pres

pres.setup_matplotlib()
display(pres.pipeline_overview_df())


,Phase,What happens,Why it matters
0,Data loading,Read cached SNAP edge list into a dataframe,"Shows dataframe-native ingest cost, not just graph runtime"
1,Data shaping,Compute node degree and materialize seed flags,Benchmarks the dataframe wrangling before any traversal
2,Graph search,Run GFQL neighborhood expansion around interesting nodes,Measures query/search on top of columnar data
3,Graph analytics,Run PageRank with igraph/cugraph or Neo4j GDS,Shows backend graph algorithm acceleration
4,Downstream use,Keep the final enriched subgraph for visualization or follow-on analysis,No external DB required; it stays in Python dataframes/graph objects


## Exact 3-way comparison on Twitter

Twitter is the cleanest exact comparison: all three engines finish comfortably,
and we can measure the full lifecycle including data loading.
The stacked bars show **ETL** (load + shape), **Search** (graph queries), and **Analytics** (PageRank).


In [2]:
pres.plot_twitter_lifecycle();


## Larger-graph story on GPlus

GPlus (30M edges) is where the story becomes especially compelling.
Neo4j becomes expensive enough that the honest result is only a lower bound.


In [5]:
pres.plot_gplus_lifecycle();


## Why this matters

- **Free OSS**: the benchmark stays in open-source Python tooling.
- **No external DB required**: the Graphistry CPU and GPU paths run directly on your dataframes / graph objects.
- **CPU path is already useful**: even without the GPU, this shows a real graph pipeline that does not require a graph database.
- **GPU path compounds the win**: both the columnar dataframe work and the graph analytics work accelerate.

That is why this is more than “GPU PageRank is fast.” It is an end-to-end graph workflow running right where your data already is.


## Reproduction notes

The benchmark commands live under `benchmarks/gfql/filter_pagerank/`, but this notebook intentionally uses only saved result JSON files.

If you want to reproduce on DGX later, the current benchmark scripts are:

- `benchmarks/gfql/filter_pagerank/filter_pagerank_pipeline_cpu_gpu.py`
- `benchmarks/gfql/filter_pagerank/load_prepare_cpu_gpu.py`
- `benchmarks/gfql/filter_pagerank/filter_pagerank_pipeline_neo4j.py`
